# Mock Data Generation — Adjustor Assignment
Generates mock data for all 4 tables:
1. **Claim and Exposure Data** — 20 claims, 28 exposure rows
2. **Assignment Data** — Event-level (Assigned / Reassigned / Escalated / Referred). LLM parses `Activity Notes` to extract adjuster name + reason.
3. **Adjuster Profile and Capability** — 15 adjusters with industry-realistic profiles
4. **Adjuster Capacity** — Workload and availability per adjuster

In [ ]:
import pandas as pd
import random
from datetime import datetime, timedelta

## Table 1: Claim and Exposure Data

In [ ]:
random.seed(42)

lob_claim_types = {
    "Auto":                ["Collision", "Comprehensive", "PIP", "UM/UIM", "Liability"],
    "Homeowners":          ["Wind/Hail", "Fire", "Water Damage", "Theft", "Liability"],
    "Workers Comp":        ["Medical Only", "Lost Time", "Permanent Disability", "Fatal"],
    "Commercial Property": ["Fire", "Wind/Hail", "Flood", "Vandalism", "Equipment Breakdown"],
    "General Liability":   ["Bodily Injury", "Property Damage", "Personal Injury", "Products Liability"],
}
coverage_types = ["Collision", "Comprehensive", "PIP", "BI", "Liability", "MedPay", "UM", "Indemnity", "Property"]
loss_causes = [
    "Collision - Rear Ended", "Collision - Fixed Object", "Collision - Head On",
    "Hit and Run", "Hail Damage", "Wind Damage", "Fire - Electrical",
    "Water Damage - Pipe Burst", "Theft - Vehicle", "Slip and Fall",
    "Premises Liability", "Vehicle Accident", "Equipment Malfunction",
]
states = ["OH", "PA", "VA", "IL", "KY", "NY", "MD", "IN", "WV", "NC", "TN", "MI", "WI"]
complexities = ["Low", "Medium", "High"]

claim_configs = [
    ("CLM-00001", 2), ("CLM-00002", 1), ("CLM-00003", 2), ("CLM-00004", 1),
    ("CLM-00005", 3), ("CLM-00006", 1), ("CLM-00007", 2), ("CLM-00008", 1),
    ("CLM-00009", 1), ("CLM-00010", 2), ("CLM-00011", 1), ("CLM-00012", 2),
    ("CLM-00013", 1), ("CLM-00014", 3), ("CLM-00015", 1), ("CLM-00016", 2),
    ("CLM-00017", 1), ("CLM-00018", 1), ("CLM-00019", 2), ("CLM-00020", 1),
]

def rand_date(start="2024-01-01", end="2025-01-01"):
    s = datetime.strptime(start, "%Y-%m-%d")
    e = datetime.strptime(end, "%Y-%m-%d")
    return s + timedelta(days=random.randint(0, (e - s).days))

rows = []
for claim_num, num_exp in claim_configs:
    lob = random.choice(list(lob_claim_types.keys()))
    claim_type = random.choice(lob_claim_types[lob])
    state = random.choice(states)
    fnol_date = rand_date()
    complexity = random.choices(complexities, weights=[0.50, 0.35, 0.15])[0]
    closed = random.random() < 0.60
    close_date = (fnol_date + timedelta(days=random.randint(20, 180))).strftime("%m/%d/%Y") if closed else ""
    reopen = ""
    if closed and random.random() < 0.10:
        reopen = (datetime.strptime(close_date, "%m/%d/%Y") + timedelta(days=random.randint(5, 30))).strftime("%m/%d/%Y")
    for exp_idx in range(1, num_exp + 1):
        rows.append({
            "Claim Number": claim_num,
            "Exposure ID": f"{claim_num}-EXP{exp_idx:02d}",
            "Line of Business": lob,
            "Claim Type": claim_type,
            "Jurisdiction/State": state,
            "FNOL Date/Date Reported": fnol_date.strftime("%m/%d/%Y"),
            "Claim Create Date": fnol_date.strftime("%m/%d/%Y"),
            "Claim Close Date": close_date,
            "Reopen Date": reopen,
            "Coverage Type": random.choice(coverage_types),
            "Loss Cause": random.choice(loss_causes),
            "Litigation Flag": random.choices(["Y", "N"], weights=[0.12, 0.88])[0],
            "CAT Flag": random.choices(["Y", "N"], weights=[0.15, 0.85])[0],
            "Subrogation Flag": random.choices(["Y", "N"], weights=[0.20, 0.80])[0],
            "Complexity": complexity,
        })

df_claims = pd.DataFrame(rows)
df_claims.to_csv("table1_claim_exposure_data.csv", index=False)
print(f"Table 1: {len(df_claims)} rows")
df_claims.head(10)

## Table 2: Assignment Data
One row per assignment event per claim. Four event types: `Assigned`, `Reassigned`, `Escalated`, `Referred`.

**LLM Task:** Parse `Activity Notes` (raw narrative text) to populate:
- `Assigned To Adjuster` — who received the claim at this step
- `Reason for Reassignment` — inferred reason for the event

In [ ]:
# Adjuster roster — must match Table 3
adjusters = [
    ("Sarah Mitchell",    "West Zone Operations Ohio Team 6",         "Auto"),
    ("James Kowalski",    "West Zone Operations Illinois Team 4",     "Auto"),
    ("Linda Reyes",       "Rochester Liability Team 2",               "Liability"),
    ("Marcus Thompson",   "Columbus Property Team 3",                 "Property"),
    ("Patricia Chen",     "Cleveland Workers Comp Team 1",            "Workers Comp"),
    ("David O'Brien",     "ISS Special Investigations Unit",          "ISS/SIU"),
    ("Anita Patel",       "West Zone Operations Ohio Team 3",         "Auto"),
    ("Robert Harrington", "Pittsburgh Property Team 2",               "Property"),
    ("Kevin Walsh",       "CAT Response Team - Central",              "CAT/Auto"),
    ("Jennifer Nguyen",   "Rochester Liability Team 4",               "Liability"),
    ("Thomas Burke",      "Parkersburg MD Team 4",                    "Material Damage"),
    ("Maria Gonzalez",    "Cleveland Workers Comp Team 3",            "Workers Comp"),
    ("Steven Park",       "Parkersburg MD Team 2",                    "Material Damage"),
    ("Rachel Coleman",    "Senior Adjusters Unit",                    "All Lines"),
    ("Christopher Adams", "Subrogation Recovery Unit",                "Subrogation"),
]

# Note templates: the LLM reads these and extracts adjuster name + reason
def make_notes(event_type, adj_name, group, context):
    if event_type == "Assigned":
        t = random.choice([
            f"Claim created and assigned to {adj_name} in group {group}. {context}",
            f"Initial FNOL intake complete. Claim routed to {adj_name} ({group}) based on loss location and LOB eligibility. {context}",
            f"New claim received. Assignment Engine routed to {adj_name} in {group}. {context}",
        ])
    elif event_type == "Reassigned":
        t = random.choice([
            f"Assigned to {adj_name} in {group}. {context}",
            f"Claim transferred to {adj_name} ({group}). {context}",
            f"Reassigned from prior adjuster to {adj_name} in {group}. {context}",
            f"Workload rebalance triggered. Claim moved to {adj_name} in {group}. {context}",
        ])
    elif event_type == "Escalated":
        t = random.choice([
            f"Claim escalated to {adj_name} in {group}. {context}",
            f"Complexity threshold exceeded. Escalated to senior handler {adj_name} in {group}. {context}",
            f"Supervisor review required. Escalation issued to {adj_name} ({group}). {context}",
        ])
    else:  # Referred
        t = random.choice([
            f"Manually referred to {adj_name} in {group}. {context}",
            f"Specialist referral issued to {adj_name} ({group}) for further review. {context}",
            f"Coverage question identified. Claim referred to {adj_name} in {group}. {context}",
        ])
    return t

action_contexts = [
    "Action: New exposure added; policy coverage verified.",
    "Action: Loss cause updated from initial FNOL report.",
    "Flag: Activity Overdue - initial contact with insured not completed.",
    "Action: Supplement inspection requested by field appraiser.",
    "Action: Date of loss corrected per police report.",
    "Action: Exposure closed; partial payment issued.",
    "Flag: Litigation hold applied; outside counsel notified.",
    "Action: CAT team engaged due to widespread weather event in region.",
    "Action: Potential ISS involvement identified during recorded statement.",
    "Action: Subrogation opportunity flagged against third-party carrier.",
    "Action: Medical records requested; treating physician identified.",
    "Action: Independent Medical Examination (IME) ordered.",
    "Flag: Coverage dispute - policy exclusion under review by coverage counsel.",
    "Action: Insured requested status update; call completed and documented.",
    "Action: Assigned back to material damage specialist for reinspection.",
    "Action: Final payment authorized; all exposures closed.",
    "Action: Claimant's attorney contacted; representation letter received.",
    "Action: Rental authorization approved; vehicle in repair shop.",
    "Action: Total loss determination initiated; market value assessed.",
    "Flag: Workload threshold exceeded on prior adjuster - system-triggered transfer.",
    "Action: Recorded statement completed; liability assessment in progress.",
    "Flag: Adjuster OOO - claim reassigned to maintain contact SLA.",
]

reason_codes = [
    "Initial Assignment",
    "Workload Rebalancing",
    "Adjuster Out of Office / PTO",
    "LOB Mismatch - Specialty Required",
    "Geographic Routing Correction",
    "Complexity Upgrade",
    "Litigation Referral",
    "SIU / ISS Escalation",
    "Coverage Review Required",
    "CAT Activation",
    "Subrogation Referral",
    "Activity Overdue - Supervisor Reassignment",
    "Functional Handoff - MD to Liability",
    "Return After Specialist Review",
]

def get_trigger(event_type):
    if event_type == "Assigned":
        return random.choices(["System", "Manual"], weights=[0.75, 0.25])[0]
    if event_type == "Reassigned":
        return random.choices(["System", "Manual"], weights=[0.45, 0.55])[0]
    return "Manual"

def get_assigned_by(trigger):
    if trigger == "System":
        return "Assignment Engine (Auto-Route)"
    return random.choice(["Rachel Coleman", "Supervisor - Team Lead", "Claims Manager"])

claim_meta = df_claims.drop_duplicates("Claim Number").set_index("Claim Number")
assignment_rows = []

for claim_num in df_claims["Claim Number"].unique():
    meta = claim_meta.loc[claim_num]
    complexity = meta["Complexity"]
    fnol_dt = datetime.strptime(meta["FNOL Date/Date Reported"], "%m/%d/%Y")
    exposures = df_claims[df_claims["Claim Number"] == claim_num]["Exposure ID"].tolist()

    num_events = {"Low": random.randint(2, 3), "Medium": random.randint(3, 5), "High": random.randint(5, 8)}[complexity]
    current_time = fnol_dt + timedelta(hours=random.randint(0, 4))
    first_adj = random.choice(adjusters)
    adj_name, adj_group = first_adj[0], first_adj[1]

    for ev_idx in range(1, num_events + 1):
        exp_id = random.choice(exposures)
        if ev_idx == 1:
            event_type = "Assigned"
            reason = "Initial Assignment"
        else:
            event_type = random.choices(
                ["Reassigned", "Escalated", "Referred"],
                weights=[0.60, 0.25, 0.15]
            )[0]
            new_adj = random.choice([a for a in adjusters if a[0] != adj_name])
            adj_name, adj_group = new_adj[0], new_adj[1]
            reason = random.choice([r for r in reason_codes if r != "Initial Assignment"])

        context = random.choice(action_contexts)
        trigger = get_trigger(event_type)

        assignment_rows.append({
            "Claim Number": claim_num,
            "Exposure ID": exp_id,
            "Assignment #": ev_idx,
            "Event Type": event_type,
            "Timestamp": current_time.strftime("%m/%d/%Y %I:%M %p"),
            "Activity Notes": make_notes(event_type, adj_name, adj_group, context),
            "Assigned To Adjuster": "",       # LLM to extract from Activity Notes
            "Reason for Reassignment": "",    # LLM to extract from Activity Notes
            "Manual/System Trigger": trigger,
            "Assigned By": get_assigned_by(trigger),
            "Complexity": complexity,
        })
        current_time += timedelta(days=random.randint(0, 5), hours=random.randint(1, 10))

df_assignments = pd.DataFrame(assignment_rows)
df_assignments.to_csv("table2_assignment_data.csv", index=False)
print(f"Table 2: {len(df_assignments)} rows")
df_assignments.head(12)

## Table 3: Adjuster Profile and Capability

In [ ]:
adjuster_profiles = [
    {
        "Adjuster ID": "ADJ-001", "Adjuster Name": "Sarah Mitchell",
        "Adjuster Role": "Claims Adjuster II",
        "Assigned Group": "West Zone Operations Ohio Team 6",
        "LOB Eligibility": "Auto; Homeowners",
        "Authority Limits": "$50,000",
        "Regions/Jurisdiction Authorization": "OH, IN, KY, WV",
        "Claim Types Handled": "Collision; Comprehensive; PIP; UM/UIM",
        "Attributes": "Bi-lingual (Spanish); Certified Xactimate User",
        "Expertise/Specializations": "Total Loss; Medical Payments",
        "Active Claim Count": 38,
    },
    {
        "Adjuster ID": "ADJ-002", "Adjuster Name": "James Kowalski",
        "Adjuster Role": "Claims Adjuster I",
        "Assigned Group": "West Zone Operations Illinois Team 4",
        "LOB Eligibility": "Auto",
        "Authority Limits": "$25,000",
        "Regions/Jurisdiction Authorization": "IL, WI, MN",
        "Claim Types Handled": "Collision; PIP; Liability",
        "Attributes": "Active Adjuster License - IL",
        "Expertise/Specializations": "Bodily Injury; Liability Settlements",
        "Active Claim Count": 42,
    },
    {
        "Adjuster ID": "ADJ-003", "Adjuster Name": "Linda Reyes",
        "Adjuster Role": "Liability Claims Specialist",
        "Assigned Group": "Rochester Liability Team 2",
        "LOB Eligibility": "Auto; General Liability; Commercial",
        "Authority Limits": "$100,000",
        "Regions/Jurisdiction Authorization": "NY, PA, NJ, CT",
        "Claim Types Handled": "Bodily Injury; Property Damage; Personal Injury; Products Liability",
        "Attributes": "JD (Paralegal Certified); Litigation Experienced",
        "Expertise/Specializations": "Litigation Management; Coverage Disputes; High-Value BI",
        "Active Claim Count": 28,
    },
    {
        "Adjuster ID": "ADJ-004", "Adjuster Name": "Marcus Thompson",
        "Adjuster Role": "Property Claims Adjuster II",
        "Assigned Group": "Columbus Property Team 3",
        "LOB Eligibility": "Homeowners; Commercial Property",
        "Authority Limits": "$75,000",
        "Regions/Jurisdiction Authorization": "OH, PA, WV, VA",
        "Claim Types Handled": "Wind/Hail; Fire; Water Damage; Vandalism",
        "Attributes": "Certified Building Inspector; IICRC Water Damage Certified",
        "Expertise/Specializations": "Structural Damage; Water Mitigation; Contents Inventory",
        "Active Claim Count": 31,
    },
    {
        "Adjuster ID": "ADJ-005", "Adjuster Name": "Patricia Chen",
        "Adjuster Role": "Workers Compensation Specialist",
        "Assigned Group": "Cleveland Workers Comp Team 1",
        "LOB Eligibility": "Workers Comp",
        "Authority Limits": "$150,000",
        "Regions/Jurisdiction Authorization": "OH, PA, IN, KY",
        "Claim Types Handled": "Medical Only; Lost Time; Permanent Disability; Fatal",
        "Attributes": "Certified WC Professional (CWCP); Nurse Case Manager Network",
        "Expertise/Specializations": "Return-to-Work Programs; Permanent Total Disability; WC Litigation",
        "Active Claim Count": 55,
    },
    {
        "Adjuster ID": "ADJ-006", "Adjuster Name": "David O'Brien",
        "Adjuster Role": "Special Investigations Unit Lead",
        "Assigned Group": "ISS Special Investigations Unit",
        "LOB Eligibility": "Auto; Homeowners; Workers Comp; General Liability",
        "Authority Limits": "$500,000",
        "Regions/Jurisdiction Authorization": "All States",
        "Claim Types Handled": "All Lines - SIU Referrals Only",
        "Attributes": "Certified Fraud Investigator (CFI); Former Law Enforcement",
        "Expertise/Specializations": "Insurance Fraud Detection; Staged Accidents; Arson Investigation",
        "Active Claim Count": 15,
    },
    {
        "Adjuster ID": "ADJ-007", "Adjuster Name": "Anita Patel",
        "Adjuster Role": "Claims Adjuster I",
        "Assigned Group": "West Zone Operations Ohio Team 3",
        "LOB Eligibility": "Auto",
        "Authority Limits": "$20,000",
        "Regions/Jurisdiction Authorization": "OH, MI",
        "Claim Types Handled": "Collision; Comprehensive; PIP",
        "Attributes": "Active Adjuster License - OH; Bilingual (Gujarati)",
        "Expertise/Specializations": "PIP Medical Review; Rental Authorization",
        "Active Claim Count": 47,
    },
    {
        "Adjuster ID": "ADJ-008", "Adjuster Name": "Robert Harrington",
        "Adjuster Role": "Property Claims Adjuster I",
        "Assigned Group": "Pittsburgh Property Team 2",
        "LOB Eligibility": "Homeowners; Commercial Property",
        "Authority Limits": "$40,000",
        "Regions/Jurisdiction Authorization": "PA, WV, OH",
        "Claim Types Handled": "Wind/Hail; Fire; Water Damage; Theft",
        "Attributes": "Xactimate Certified; Field Inspector",
        "Expertise/Specializations": "Roof Inspections; Hail Damage Assessment",
        "Active Claim Count": 36,
    },
    {
        "Adjuster ID": "ADJ-009", "Adjuster Name": "Kevin Walsh",
        "Adjuster Role": "CAT Claims Specialist",
        "Assigned Group": "CAT Response Team - Central",
        "LOB Eligibility": "Auto; Homeowners; Commercial Property",
        "Authority Limits": "$200,000",
        "Regions/Jurisdiction Authorization": "OH, IN, KY, TN, IL, WI",
        "Claim Types Handled": "Wind/Hail; Flood; Fire; Collision (CAT)",
        "Attributes": "HAAG Certified; CAT Deployment Ready; FAA Part 107 Drone Certified",
        "Expertise/Specializations": "Catastrophe Response; Large Loss; Agricultural Claims",
        "Active Claim Count": 82,
    },
    {
        "Adjuster ID": "ADJ-010", "Adjuster Name": "Jennifer Nguyen",
        "Adjuster Role": "Liability Claims Specialist",
        "Assigned Group": "Rochester Liability Team 4",
        "LOB Eligibility": "Auto; General Liability",
        "Authority Limits": "$75,000",
        "Regions/Jurisdiction Authorization": "NY, NJ, PA, CT, MA",
        "Claim Types Handled": "Bodily Injury; Premises Liability; Products Liability",
        "Attributes": "Litigation Experienced; Mediation Certified",
        "Expertise/Specializations": "High-Value Bodily Injury; Premises Liability; UM/UIM",
        "Active Claim Count": 22,
    },
    {
        "Adjuster ID": "ADJ-011", "Adjuster Name": "Thomas Burke",
        "Adjuster Role": "Material Damage Specialist",
        "Assigned Group": "Parkersburg MD Team 4",
        "LOB Eligibility": "Auto",
        "Authority Limits": "$60,000",
        "Regions/Jurisdiction Authorization": "WV, VA, MD, OH, PA",
        "Claim Types Handled": "Collision; Comprehensive; Total Loss; Supplement Review",
        "Attributes": "ASE Certified; DRP Network Manager; Audatex Certified",
        "Expertise/Specializations": "Total Loss Valuation; OEM Parts Disputes; Body Shop Negotiations",
        "Active Claim Count": 44,
    },
    {
        "Adjuster ID": "ADJ-012", "Adjuster Name": "Maria Gonzalez",
        "Adjuster Role": "Workers Compensation Adjuster II",
        "Assigned Group": "Cleveland Workers Comp Team 3",
        "LOB Eligibility": "Workers Comp",
        "Authority Limits": "$75,000",
        "Regions/Jurisdiction Authorization": "OH, IN, PA",
        "Claim Types Handled": "Medical Only; Lost Time; Vocational Rehabilitation",
        "Attributes": "Bilingual (Spanish); Certified Rehabilitation Counselor (CRC)",
        "Expertise/Specializations": "Vocational Rehab; RTW Coordination; Pharmacy Benefit Management",
        "Active Claim Count": 60,
    },
    {
        "Adjuster ID": "ADJ-013", "Adjuster Name": "Steven Park",
        "Adjuster Role": "Material Damage Specialist",
        "Assigned Group": "Parkersburg MD Team 2",
        "LOB Eligibility": "Auto",
        "Authority Limits": "$50,000",
        "Regions/Jurisdiction Authorization": "WV, VA, MD, KY",
        "Claim Types Handled": "Collision; Comprehensive; Hit and Run; Total Loss",
        "Attributes": "Audatex Certified; ARMS Platform Trained",
        "Expertise/Specializations": "Salvage Title Processing; Diminished Value Assessment",
        "Active Claim Count": 39,
    },
    {
        "Adjuster ID": "ADJ-014", "Adjuster Name": "Rachel Coleman",
        "Adjuster Role": "Senior Claims Supervisor",
        "Assigned Group": "Senior Adjusters Unit",
        "LOB Eligibility": "Auto; Homeowners; General Liability; Workers Comp; Commercial",
        "Authority Limits": "$500,000",
        "Regions/Jurisdiction Authorization": "All States",
        "Claim Types Handled": "All Lines - Oversight and Escalation",
        "Attributes": "SCLA Designation; Team Lead; Quality Review Authority",
        "Expertise/Specializations": "Complex Claims Oversight; Coverage Analysis; Team Mentoring",
        "Active Claim Count": 10,
    },
    {
        "Adjuster ID": "ADJ-015", "Adjuster Name": "Christopher Adams",
        "Adjuster Role": "Subrogation Recovery Specialist",
        "Assigned Group": "Subrogation Recovery Unit",
        "LOB Eligibility": "Auto; Homeowners; Commercial Property",
        "Authority Limits": "$250,000",
        "Regions/Jurisdiction Authorization": "All States",
        "Claim Types Handled": "Subrogation Referrals - All Lines",
        "Attributes": "CPCU Designation; Inter-Company Arbitration Certified (Arbitration Forums)",
        "Expertise/Specializations": "Third-Party Recovery; Inter-Company Arbitration; Lien Management",
        "Active Claim Count": 20,
    },
]

df_profiles = pd.DataFrame(adjuster_profiles)
df_profiles.to_csv("table3_adjuster_profile_capability.csv", index=False)
print(f"Table 3: {len(df_profiles)} rows")
df_profiles

## Table 4: Adjuster Capacity

In [ ]:
max_capacities = {
    "ADJ-001": 50, "ADJ-002": 55, "ADJ-003": 40, "ADJ-004": 45,
    "ADJ-005": 65, "ADJ-006": 20, "ADJ-007": 55, "ADJ-008": 50,
    "ADJ-009": 90, "ADJ-010": 35, "ADJ-011": 55, "ADJ-012": 70,
    "ADJ-013": 50, "ADJ-014": 15, "ADJ-015": 30,
}

def calc_weighted_workload(n):
    """Weighted workload: High=3pts, Medium=2pts, Low=1pt (20/40/40 mix assumption)."""
    h, m = round(n * 0.20), round(n * 0.40)
    return h * 3 + m * 2 + (n - h - m) * 1

def get_availability(active, max_cap, pto):
    if pto == "Y":
        return "Unavailable - PTO"
    pct = active / max_cap
    if pct < 0.60: return "Available"
    if pct < 0.80: return "Near Capacity"
    if pct < 1.00: return "At Capacity"
    return "Overloaded"

capacity_rows = []
for p in adjuster_profiles:
    adj_id = p["Adjuster ID"]
    active = p["Active Claim Count"]
    max_cap = max_capacities[adj_id]
    pto = random.choices(["Y", "N"], weights=[0.10, 0.90])[0]
    capacity_rows.append({
        "Adjuster ID": adj_id,
        "Adjuster Name": p["Adjuster Name"],
        "Active Open Claims": active,
        "Max Claim Capacity": max_cap,
        "Weighted Workload Score": calc_weighted_workload(active),
        "Availability Status": get_availability(active, max_cap, pto),
        "PTO/OOO Flag": pto,
    })

df_capacity = pd.DataFrame(capacity_rows)
df_capacity.to_csv("table4_adjuster_capacity.csv", index=False)
print(f"Table 4: {len(df_capacity)} rows")
df_capacity

## Sample — LLM Parsing Preview
The `Activity Notes` column is the raw input for the LLM.  
The two blank columns are what the LLM should output after parsing each note.

In [ ]:
# Show one claim's full assignment history — demonstrates the LLM parsing task
pd.set_option("display.max_colwidth", 120)
sample = df_assignments[df_assignments["Claim Number"] == "CLM-00005"].copy()
cols = ["Claim Number", "Assignment #", "Event Type", "Timestamp",
        "Activity Notes", "Assigned To Adjuster", "Reason for Reassignment",
        "Manual/System Trigger", "Assigned By"]
sample[cols]